# Lesson 16: Context engineering and memory

*Companion notebook for [the lesson page](../lessons/16-context-engineering.html). Worked solutions are in `solutions/16-context-engineering-solutions.ipynb`; try each exercise first.*

Decide what the model sees on every call: measure how budgets, position and distractors affect quality, keep long conversations and agent runs within budget with selection, compression, ordering, isolation and caching, then give assistants memory that persists across sessions without leaking or being poisoned.

**Runs on:** Any computer (CPU). Everything runs offline with fake models; cells that call a real model are marked and need an API key.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q anthropic python-dotenv numpy matplotlib

In [ ]:
import os, json, re, random
import anthropic
from dotenv import load_dotenv

load_dotenv()   # reads ANTHROPIC_API_KEY from a .env file next to this notebook, if there is one
# On Google Colab, add the key under the key icon (Secrets), then uncomment the next two lines:
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()         # only used in cells marked "Needs an API key"
MODEL = "claude-sonnet-5-5"            # check your provider's current model list
CHEAP_MODEL = "claude-haiku-5-5"       # for summaries and bulk harness calls
print("API key found" if os.environ.get("ANTHROPIC_API_KEY") else
      "No API key set: cells marked 'Needs an API key' will fail until you add one")

## What the model sees is all it knows

### Counting tokens without an API

The only exact count comes from the provider's tokenizer for the model you call (`count_tokens`, lesson "LLM APIs"); a count for one model is only an estimate for another. Offline, a rule of thumb is enough to plan with: about four characters per token for English prose. Code, JSON, numbers and non-English text need more tokens per character, so the rule undercounts them.

In [ ]:
CHARS_PER_TOKEN = 4.0      # rule of thumb for English prose; code and other languages need more tokens

def approx_tokens(text: str) -> int:
    """Rough token estimate. Expect errors of tens of percent on code, JSON and non-English text."""
    return max(1, round(len(text) / CHARS_PER_TOKEN))

samples = {
    "prose": "The instrument stopped logging overnight after a failed upload and needs a restart.",
    "code":  "def restart(instrument_id: str) -> dict:\n    return {'id': instrument_id, 'ok': True}",
    "json":  '{"id": "P-117", "site": "coastal", "status": "offline", "firmware": "2.3"}',
}
for name, text in samples.items():
    print(f"{name:6s} {len(text):3d} chars  ~{approx_tokens(text):3d} tokens")

Expected output:

```text
prose   83 chars  ~ 21 tokens
code    85 chars  ~ 21 tokens
json    74 chars  ~ 18 tokens
```

Check the rule against the real counter once, on your own kind of text, then trust the estimate for budgeting. Its error is tens of percent: fine for a budget, useless for a hard limit, so keep a margin.

In [ ]:
# Needs an API key (see the setup cell)
def real_tokens(text: str) -> int:
    """Exact count from the provider, for the model you will actually call."""
    resp = client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": text}])
    return resp.input_tokens

for name, text in samples.items():
    real = real_tokens(text)
    print(f"{name:6s} estimate {approx_tokens(text):3d}  real {real:3d}  ratio {approx_tokens(text) / real:.2f}")
# The count includes a few tokens of message framing, so short texts look over-counted.
# Calibrate CHARS_PER_TOKEN on a few kilobytes of your own data, then keep using the estimate offline.

### A budget report

Make the breakdown routine. A function that prints what each component of a request costs, as tokens and as a share of your budget, is worth more than any rule about "long prompts". Here the components are synthetic text of realistic sizes.

In [ ]:
def budget_report(parts: dict[str, str], budget: int, reply_reserve: int, count=approx_tokens) -> int:
    """Print what each component of a request costs in tokens, and whether the whole fits."""
    counts = {name: count(text) for name, text in parts.items()}
    total = sum(counts.values())
    print(f"{'component':24s} {'tokens':>7s} {'share':>7s}")
    for name, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"{name:24s} {n:7,d} {n / budget:7.1%}")
    print(f"{'total input':24s} {total:7,d} {total / budget:7.1%}")
    room = budget - reply_reserve - total
    print("fits" if room >= 0 else "OVER BUDGET", f"({room:,d} tokens of room before the reply reserve)")
    return total

def fake_text(n_tokens: int, sentence: str) -> str:
    """Repeat a sentence until the estimate reaches n_tokens: a stand-in for real content."""
    return " ".join([sentence] * round(n_tokens * CHARS_PER_TOKEN / (len(sentence) + 1)))

TOOL = {"name": "get_instrument_status",
        "description": "Look up the current status of one field instrument by its id (format 'P-123'). "
                       "Returns site, status (ok/degraded/offline), time of last data (UTC) and firmware.",
        "input_schema": {"type": "object", "properties": {"instrument_id": {"type": "string"}},
                         "required": ["instrument_id"]}}
parts = {
    "system prompt":        fake_text(600, "Answer from the manual and the tool results; cite sections."),
    "tool definitions (6)": json.dumps([TOOL] * 6),
    "memory profile":       fake_text(300, "User Dana runs the coastal site and prefers metric units."),
    "retrieved chunks (8)": fake_text(4_000, "Section 4.2: a logging stall after a failed upload needs a restart."),
    "conversation history": fake_text(6_000, "user: is P-117 healthy? assistant: it went offline at 02:00 UTC."),
    "tool results":         fake_text(3_000, '{"id": "P-117", "status": "offline", "firmware": "2.3"}'),
    "question":             "Which coastal instruments went offline this week, and what should I check first?",
}
BUDGET = 32_000          # what you choose to spend per request, well below the model's window (see the docs)
REPLY_RESERVE = 2_000    # room for the answer: input plus output must fit in the window
total = budget_report(parts, BUDGET, REPLY_RESERVE)

Expected output:

```text
component                 tokens   share
conversation history       5,996   18.7%
retrieved chunks (8)       3,995   12.5%
tool results               2,996    9.4%
system prompt                600    1.9%
tool definitions (6)         501    1.6%
memory profile               304    0.9%
question                      20    0.1%
total input               14,412   45.0%
fits (15,588 tokens of room before the reply reserve)
```

### How models use long context

### Measure it yourself: a needle in a distractor-rich haystack

The harness below generates a log of instrument events, hides one line (the needle) at a chosen depth, asks a question only that line answers, and scores the reply by exact match. One line in five is a near miss: the same instrument doing something else, or another instrument with its own calibration constant.

In [ ]:
INSTRUMENTS = [f"P-{n}" for n in range(100, 400) if n != 117]   # P-117 is reserved for the needle
EVENTS = ["firmware was updated to 2.{v}", "battery was replaced", "logged {n} records",
          "was moved to the {site} site", "reported a supply voltage of 1{v}.{n} V"]
NEEDLE = "On 2026-04-02 instrument P-117 was assigned the calibration constant 0.9137."
QUESTION = "What calibration constant was assigned to instrument P-117? Reply with the number only."
ANSWER = "0.9137"

def distractor(rng: random.Random) -> str:
    """A log line shaped like the needle. One in five is a near miss: same instrument or same event."""
    kind, inst = rng.random(), rng.choice(INSTRUMENTS)
    date = f"2026-0{rng.randint(1, 9)}-{rng.randint(10, 28)}"
    if kind < 0.1:                                    # same instrument, different event
        inst = "P-117"
    elif kind < 0.2:                                  # same event, different instrument and number
        return f"On {date} instrument {inst} was assigned the calibration constant 0.{rng.randint(1000, 9999)}."
    event = rng.choice(EVENTS).format(v=rng.randint(1, 5), n=rng.randint(100, 999),
                                      site=rng.choice(["urban", "alpine", "coastal"]))
    return f"On {date} instrument {inst} {event}."

def make_haystack(n_lines: int, depth: float, rng: random.Random) -> str:
    lines = [distractor(rng) for _ in range(n_lines)]
    lines.insert(int(depth * n_lines), NEEDLE)        # depth 0.0 = first line, 1.0 = last line
    return "\n".join(lines)

def run_grid(answer_fn, sizes, depths, trials=3, seed=0):
    """Accuracy of answer_fn(prompt) for each haystack size (lines) and needle depth."""
    rng = random.Random(seed)
    print("lines   tokens  " + "  ".join(f"d={d:.2f}" for d in depths))
    for n in sizes:
        hits = []
        for d in depths:
            ok = 0
            for _ in range(trials):
                prompt = f"<log>\n{make_haystack(n, d, rng)}\n</log>\n\n{QUESTION}"
                ok += ANSWER in answer_fn(prompt)
            hits.append(ok / trials)
        print(f"{n:5d} {approx_tokens(prompt):8,d}  " + "  ".join(f"{h:6.2f}" for h in hits))

Test the harness before you trust it. A perfect reader must score 1.0 everywhere, or the harness is broken (an early version of this one let a distractor collide with the needle). A reader that silently truncates its input shows the shape of failure the grid can detect.

In [ ]:
def grep_model(prompt: str) -> str:
    """A perfect reader: finds the sentence with a regular expression."""
    m = re.search(r"P-117 was assigned the calibration constant ([0-9.]+)", prompt)
    return m.group(1) if m else "not found"

def truncating_model(prompt: str, limit: int = 30_000) -> str:
    """A reader that silently drops everything after `limit` characters."""
    return grep_model(prompt[:limit])

print("perfect reader"); run_grid(grep_model, sizes=[200, 800], depths=[0.0, 0.5, 1.0])
print("truncating reader"); run_grid(truncating_model, sizes=[200, 800, 2_000], depths=[0.0, 0.5, 1.0])

Expected output:

```text
perfect reader
lines   tokens  d=0.00  d=0.50  d=1.00
  200    3,118    1.00    1.00    1.00
  800   12,118    1.00    1.00    1.00
truncating reader
lines   tokens  d=0.00  d=0.50  d=1.00
  200    3,118    1.00    1.00    1.00
  800   12,118    1.00    1.00    0.00
 2000   30,422    1.00    0.00    0.00
```

The same grid as a picture. A real model's heatmap is softer than this cliff; the point is to have a picture to compare against.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sizes, depths = [200, 400, 800, 1_600, 3_200], [0.0, 0.25, 0.5, 0.75, 1.0]
rng = random.Random(1)
acc = np.zeros((len(sizes), len(depths)))
for i, n in enumerate(sizes):
    for j, d in enumerate(depths):
        acc[i, j] = sum(ANSWER in truncating_model(f"<log>\n{make_haystack(n, d, rng)}\n</log>\n\n{QUESTION}")
                        for _ in range(3)) / 3

fig, ax = plt.subplots(figsize=(5.5, 3.4))
im = ax.imshow(acc, vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(depths)), [f"{d:.2f}" for d in depths])
ax.set_yticks(range(len(sizes)), [f"{n:,}" for n in sizes])
ax.set_xlabel("needle depth (0 = first line, 1 = last line)")
ax.set_ylabel("haystack size (lines)")
ax.set_title("Truncating reader: accuracy by size and depth")
fig.colorbar(im, label="accuracy")
plt.tight_layout()
plt.show()

Now the real thing. Real failures are softer than the truncating reader's cliff: accuracy drifts down as the haystack grows or near misses multiply, and it varies between runs, so use several trials per cell and change one variable at a time.

In [ ]:
# Needs an API key (see the setup cell)
def claude_reader(prompt: str) -> str:
    resp = client.messages.create(model=CHEAP_MODEL, max_tokens=300, output_config={"effort": "low"},
                                  messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in resp.content if b.type == "text")

run_grid(claude_reader, sizes=[200, 2_000], depths=[0.0, 0.5, 1.0], trials=2)
# 12 calls. Look at the row for 2,000 lines: does accuracy change with depth? Then raise the size until it does,
# and increase the near-miss share in distractor() to see which hurts more, length or similarity.

### Hands-on: two needles and a cost column

1. Add a second needle (a service ticket for P-117) at its own depth, and a question that needs both facts. Score a reply as correct only if it contains both answers.
2. Add a column with the estimated input tokens per call and its cost at your model's input price, kept in a named constant.
3. Check the grid offline with the perfect and truncating readers, using depth pairs such as (0.1, 0.9), (0.45, 0.55) and (0.9, 0.1).
4. With an API key, run it with the real model at two sizes and note which depth pair fails first, and what each row costs.

In [ ]:
# Your code here

## Four moves: select, compress, order, isolate

### Select

Retrieval (lesson "RAG") is selection applied to documents: five relevant chunks rather than the manual. The same idea applies to tool results, the fastest-growing part of an agent's context. A tool that returns 300 rows of JSON when the model asked about offline instruments is a selection failure. Fix it in the tool: return the fields the model needs, cap the rows, and say what was left out so the model can ask for more.

In [ ]:
def compact_rows(rows: list[dict], keep: tuple[str, ...], max_rows: int = 20) -> dict:
    """Return only the fields the model needs, cap the row count, and say what was left out."""
    shown = [{k: r[k] for k in keep if k in r} for r in rows[:max_rows]]
    out = {"rows": shown, "returned": len(shown), "total": len(rows)}
    if len(rows) > max_rows:
        out["note"] = f"{len(rows) - max_rows} more rows omitted; filter or paginate to see them"
    return out

fleet = [{"id": f"P-{n}", "site": "coastal" if n % 3 else "urban", "status": "ok" if n % 7 else "offline",
          "firmware": "2.4", "last_data": "2026-10-19T09:55Z", "notes": "routine inspection " * 10}
         for n in range(100, 400)]
raw = json.dumps(fleet)
offline = [r for r in fleet if r["status"] == "offline"]
small = json.dumps(compact_rows(offline, keep=("id", "site", "last_data")))
print(f"raw tool result: ~{approx_tokens(raw):,} tokens  ->  filtered and trimmed: ~{approx_tokens(small):,} tokens")
print(small[:160] + "...")

Expected output:

```text
raw tool result: ~23,104 tokens  ->  filtered and trimmed: ~372 tokens
{"rows": [{"id": "P-105", "site": "urban", "last_data": "2026-10-19T09:55Z"}, {"id": "P-112", "site": "coastal", "last_data": "2026-10-19T09:55Z"}, {"id": "P-11...
```

### Compress

Providers are starting to do some of this server-side: the Claude API has a beta compaction feature that summarises earlier context past a threshold and returns a block you must keep in the history, and a context-editing beta that clears old tool results. Names and limits change, so check the documentation.

In [ ]:
# Needs an API key (see the setup cell)
# Server-side compaction (beta at the time of writing: check the docs for current names and limits).
# The API summarises earlier context into a compaction block once the conversation grows past a threshold.
messages = [{"role": "user", "content": "In one sentence, what is context engineering?"}]
resp = client.beta.messages.create(
    model=MODEL, max_tokens=1_000,
    betas=["compact-2026-01-12"],
    context_management={"edits": [{"type": "compact_20260112"}]},
    messages=messages,
)
messages.append({"role": "assistant", "content": resp.content})   # keep the whole content, not just the text
print([b.type for b in resp.content])
# A related beta, context editing (betas=["context-management-2025-06-27"] with
# {"type": "clear_tool_uses_20250919"}), clears old tool results instead of summarising them.

### Order and structure

### Isolate

## Prompt caching: lay out the prompt so the cache hits

In a healthy multi-turn loop the stable prefix is written once and read on every later turn, each turn's new content is written once and read thereafter, and only the newest tail is processed at full price. The calculator shows the difference.

In [ ]:
PRICE_IN = 2.0            # USD per million input tokens: set from your provider's pricing page
CACHE_WRITE_MULT = 1.25   # cost of writing the cache (default 5-minute lifetime), relative to PRICE_IN
CACHE_READ_MULT = 0.10    # cost of reading it; lower on some models, so check the docs

def agent_run_cost(turns: int, prefix_tokens: int, tokens_per_turn: int, cached: bool) -> float:
    """Input cost of a loop where every turn resends the stable prefix and the growing history."""
    cost = 0.0
    for t in range(turns):
        history, new = t * tokens_per_turn, tokens_per_turn
        if not cached:
            cost += (prefix_tokens + history + new) * PRICE_IN / 1e6
            continue
        prefix_mult = CACHE_WRITE_MULT if t == 0 else CACHE_READ_MULT      # written once, then read
        cost += prefix_tokens * prefix_mult * PRICE_IN / 1e6
        cost += history * CACHE_READ_MULT * PRICE_IN / 1e6                  # earlier turns were cached as they arrived
        cost += new * CACHE_WRITE_MULT * PRICE_IN / 1e6                     # this turn's addition is written
    return cost

for turns in (5, 20, 50):
    u = agent_run_cost(turns, prefix_tokens=8_000, tokens_per_turn=1_500, cached=False)
    c = agent_run_cost(turns, prefix_tokens=8_000, tokens_per_turn=1_500, cached=True)
    print(f"{turns:3d} turns: uncached {u:6.3f} USD   cached {c:6.3f} USD   saving {1 - c / u:4.0%}")

Expected output:

```text
  5 turns: uncached  0.125 USD   cached  0.048 USD   saving  61%
 20 turns: uncached  0.950 USD   cached  0.182 USD   saving  81%
 50 turns: uncached  4.625 USD   cached  0.653 USD   saving  86%
```

Cost against the number of turns, cached and uncached. The gap widens because the uncached run re-pays for the whole history on every turn.

In [ ]:
turn_counts = list(range(1, 61))
uncached = [agent_run_cost(t, 8_000, 1_500, cached=False) for t in turn_counts]
cached = [agent_run_cost(t, 8_000, 1_500, cached=True) for t in turn_counts]

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(turn_counts, uncached, label="uncached")
ax.plot(turn_counts, cached, label="cached (prefix + history)")
ax.set_xlabel("turns in the agent run")
ax.set_ylabel("input cost, USD")
ax.set_title("Prompt caching on a growing agent loop")
ax.legend()
plt.tight_layout()
plt.show()

The layout that achieves it: one explicit breakpoint at the end of the stable system prefix, plus an automatic breakpoint that follows the growing history. Watch the usage fields to confirm it works.

In [ ]:
# Needs an API key (see the setup cell)
MANUAL = "\n".join(f"Section {i}: {fake_text(40, f'Procedure {i} for the P-series instruments, step by step.')}"
                   for i in range(1, 41))                 # a long, identical-every-call reference
STABLE_SYSTEM = ("You are an operations assistant for a fleet of field instruments. Answer from the manual "
                 "below and cite section numbers.\n\n<manual>\n" + MANUAL + "\n</manual>")

def ask_cached(history: list[dict]):
    resp = client.messages.create(
        model=MODEL, max_tokens=600,
        system=[{"type": "text", "text": STABLE_SYSTEM, "cache_control": {"type": "ephemeral"}}],  # breakpoint 1
        cache_control={"type": "ephemeral"},        # automatic breakpoint on the last block: the growing history
        messages=history,
    )
    u = resp.usage
    print(f"uncached {u.input_tokens:5d}   cache write {u.cache_creation_input_tokens:5d}   "
          f"cache read {u.cache_read_input_tokens:5d}")
    return resp

history = [{"role": "user", "content": "Which section covers procedure 7? One sentence."}]
r1 = ask_cached(history)                    # first call: a cache write, little or no read
history += [{"role": "assistant", "content": r1.content},
            {"role": "user", "content": "And procedure 12? One sentence."}]
r2 = ask_cached(history)                    # within the lifetime: the prefix is read, only the new turn is written
# If cache read stays 0, something before the breakpoint changes between calls: a date, an id, an unsorted dict.

## A ContextManager under a token budget

Build it with a fake summariser first. The fake is deterministic, so the manager's behaviour can be asserted, and the only thing left to trust later is the real summariser's prompt.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable

def fake_summarise(previous: str, turns: list[dict]) -> str:
    """Deterministic stand-in for a model: keep the first eight words of each folded turn."""
    clauses = [f"{t['role']}: {' '.join(t['content'].split()[:8])}" for t in turns]
    return ((previous + " ") if previous else "") + "; ".join(clauses) + "."

@dataclass
class ContextManager:
    """Keep a conversation under a token budget: pinned facts + rolling summary + recent turns."""
    budget: int                                   # input tokens allowed for the whole request
    reply_reserve: int = 1_000                    # left free for the model's answer
    min_recent: int = 4                           # turns always kept verbatim
    summarise: Callable = fake_summarise          # (previous_summary, turns) -> new summary
    count: Callable = approx_tokens
    system: str = ""                              # stable instructions: identical every call, cacheable
    pinned: list[str] = field(default_factory=list)
    summary: str = ""
    turns: list[dict] = field(default_factory=list)
    folds: int = 0

    def pin(self, fact: str) -> None:
        self.pinned.append(fact)

    def context_block(self) -> str:
        out = ""
        if self.pinned:
            out += "<pinned_facts>\n" + "\n".join(f"- {p}" for p in self.pinned) + "\n</pinned_facts>\n"
        if self.summary:
            out += "<summary_of_earlier_conversation>\n" + self.summary + "\n</summary_of_earlier_conversation>\n"
        return out

    def system_blocks(self) -> list[dict]:
        blocks = [{"type": "text", "text": self.system, "cache_control": {"type": "ephemeral"}}]
        if self.context_block():
            blocks.append({"type": "text", "text": self.context_block()})     # changes only when we fold
        return blocks

    def tokens(self) -> int:
        return self.count(self.system) + self.count(self.context_block()) + sum(self.count(t["content"]) for t in self.turns)

    def add(self, role: str, text: str) -> None:
        self.turns.append({"role": role, "content": text})
        self.fit()

    def fit(self) -> None:
        """Fold the oldest user/assistant pair into the summary until the request fits."""
        while self.tokens() > self.budget - self.reply_reserve and len(self.turns) > self.min_recent:
            oldest, self.turns = self.turns[:2], self.turns[2:]
            self.summary = self.summarise(self.summary, oldest)
            self.folds += 1

    def report(self) -> None:
        budget_report({"system": self.system, "pinned + summary": self.context_block(),
                       "recent turns": "".join(t["content"] for t in self.turns)},
                      self.budget, self.reply_reserve, self.count)

Pinned facts and summary go into a second system block, after the stable rules block that carries the cache breakpoint (harmless if the rules are shorter than the cacheable minimum, as in this toy example). They change only at a fold, so most turns still hit the cache; a fold invalidates it from that point on, which is the price of compression. Folding whole user/assistant pairs keeps the history starting with a user turn, as the API requires.

In [ ]:
RULES = "You are a precise assistant for a field engineer. Use the pinned facts and the summary as if you remembered them."
cm = ContextManager(budget=900, reply_reserve=200, system=RULES)
cm.pin("User is Dana at the coastal site CST-7; prefers metric units and short answers.")

for i in range(12):
    cm.add("user", f"Turn {i}: what is the status of instrument P-{100 + i}, and when did it last report?")
    cm.add("assistant", f"Instrument P-{100 + i} is {'ok' if i % 2 else 'offline'}; its last data arrived "
                        f"at 0{i % 10}:00 UTC. " + "Nothing else in the log looks unusual for this unit. " * 2)
    assert cm.tokens() <= cm.budget - cm.reply_reserve, "over budget after a turn"
    assert cm.turns[0]["role"] == "user", "history must start with a user turn"

assert cm.folds > 0 and len(cm.turns) >= cm.min_recent
assert cm.turns[-1]["content"].startswith("Instrument P-111")      # recent turns are kept verbatim
assert "Turn 0" in cm.summary and "Turn 1" in cm.summary           # early turns survive in the summary
assert "Dana" in cm.context_block()                                 # pinned facts are never dropped
print(f"{cm.folds} folds, {len(cm.turns)} recent turns kept, summary ~{approx_tokens(cm.summary)} tokens")
cm.report()

Expected output:

```text
4 folds, 16 recent turns kept, summary ~108 tokens
component                 tokens   share
recent turns                 490   54.4%
pinned + summary             154   17.1%
system                        28    3.1%
total input                  672   74.7%
fits (28 tokens of room before the reply reserve)
```

The fake summariser only appends, so its summary grows; a real one rewrites the whole summary and keeps it bounded. Wire in the real model and ask about a fact that has long left the verbatim window.

In [ ]:
# Needs an API key (see the setup cell)
SUMMARY_PROMPT = """Update the running summary of a conversation. Keep every concrete fact, decision,
number and open question; drop pleasantries. Reply with the new summary only, at most 120 words.

<previous_summary>
{previous}
</previous_summary>

<new_turns>
{turns}
</new_turns>"""

def real_summarise(previous: str, turns: list[dict]) -> str:
    rendered = "\n".join(f"{t['role']}: {t['content']}" for t in turns)
    resp = client.messages.create(model=CHEAP_MODEL, max_tokens=400,
                                  messages=[{"role": "user", "content": SUMMARY_PROMPT.format(
                                      previous=previous or "(none yet)", turns=rendered)}])
    return "".join(b.text for b in resp.content if b.type == "text").strip()

live = ContextManager(budget=2_500, reply_reserve=500, summarise=real_summarise, system=RULES)
live.pin("User is Dana at the coastal site CST-7; prefers metric units and short answers.")

def chat(text: str) -> str:
    live.add("user", text)
    resp = client.messages.create(model=MODEL, max_tokens=800, system=live.system_blocks(), messages=live.turns)
    reply = "".join(b.text for b in resp.content if b.type == "text")
    live.add("assistant", reply)
    return reply

chat("The calibration constant for P-117 is 0.9137; please keep that in mind.")
for i in range(6):
    chat(f"Give me a two-sentence checklist for inspecting the battery on instrument P-{200 + i}.")
print(chat("What calibration constant did I give you for P-117?"))     # should come back from the summary
print(f"{live.folds} folds; summary: {live.summary[:300]}")

### Hands-on: clear old tool results before summarising

Summarising costs a model call and loses detail; stubbing out an old tool result costs nothing and loses only text the agent has already used. Extend the manager for agent transcripts.

1. Subclass `ContextManager` as `AgentContextManager` with an `add_tool_result(text)` method that stores the turn with a `kind` of `"tool_result"`.
2. Override `fit()`: while over budget, first replace the oldest intact tool result outside the most recent `keep_tool_results` turns with a stub such as `[tool result cleared: ~1,200 tokens]`; only when nothing is left to clear, fold the oldest pair into the summary as before.
3. Script an agent run of 15 steps with big tool results and compare the number of summariser calls with the plain manager under the same budget. Assert that the newest tool result is intact and that the result fits.

In [ ]:
# Your code here

## Memory for assistants and agents

### Storage designs

### Write policy: what is worth remembering

Remember what is **stable** (preferences, roles, sites, constraints), what the user **asked** you to remember, and **decisions** with their reasons. Do not remember what is transient, vague or secret: credentials never belong in memory. Write at natural points (the end of a session, an explicit "remember this", or when the model judges a fact durable), and always with the user's knowledge: say what was saved, and let them see, edit and delete it. A filter in code catches the obvious cases before the model's judgement is needed.

In [ ]:
SECRET_PATTERNS = [r"sk-[A-Za-z0-9_-]{8,}", r"(?i)\b(password|passcode|token)\b\s*[:=]?\s*\S+", r"\b\d{13,19}\b"]
TRANSIENT = ("right now", "today", "this morning", "currently", "at the moment")

def storable(fact: str) -> tuple[bool, str]:
    """Decide whether a candidate memory is worth keeping, and say why not if it is not."""
    if any(re.search(p, fact) for p in SECRET_PATTERNS):
        return False, "looks like a secret: never store credentials"
    if any(w in fact.lower() for w in TRANSIENT):
        return False, "transient: true today, stale next week"
    if len(fact.split()) < 4:
        return False, "too vague to be useful later"
    return True, "ok"

candidates = ["Dana prefers metric units and short answers.",
              "The API key is sk-abc123def456ghi789.",
              "Instrument P-117 is offline right now.",
              "Prefers short.",
              "Site CST-7 runs firmware 2.4 on every P-series unit."]
for c in candidates:
    ok, why = storable(c)
    print(f"{'keep' if ok else 'skip':4s}  {why:44s} {c}")

Expected output:

```text
keep  ok                                           Dana prefers metric units and short answers.
skip  looks like a secret: never store credentials The API key is sk-abc123def456ghi789.
skip  transient: true today, stale next week       Instrument P-117 is offline right now.
skip  too vague to be useful later                 Prefers short.
keep  ok                                           Site CST-7 runs firmware 2.4 on every P-series unit.
```

### Read policy: load at start or recall on demand

### Updating and forgetting

### Risks: privacy and injection

### Does memory help? Measure it

## Hands-on: a file-based memory tool

The simplest useful memory is a folder of notes the model can list, read, write and edit through a tool. The implementation has one security job that is not optional: every path the model supplies must resolve to a file inside the folder. A model that can be talked into reading `../.env` leaks your keys.

In [ ]:
from pathlib import Path

class MemoryFS:
    """Notes in one folder. Every path the model supplies must resolve to a file inside that folder."""

    def __init__(self, root: str):
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)

    def _safe(self, rel: str) -> Path:
        p = (self.root / rel).resolve()                     # resolves "..", symlinks and absolute paths
        if p == self.root or not p.is_relative_to(self.root):
            raise ValueError(f"path {rel!r} is outside the memory folder")
        return p

    def view(self, rel: str = "") -> str:
        if rel in ("", "/", "."):
            names = sorted(p.name for p in self.root.iterdir() if p.is_file())
            return "\n".join(names) if names else "(no notes yet)"
        return self._safe(rel).read_text(encoding="utf-8")

    def create(self, rel: str, text: str) -> str:
        self._safe(rel).write_text(text, encoding="utf-8")
        return f"saved {rel} ({len(text)} chars)"

    def update(self, rel: str, old: str, new: str) -> str:
        p = self._safe(rel)
        text = p.read_text(encoding="utf-8")
        if text.count(old) != 1:
            raise ValueError(f"'{old}' occurs {text.count(old)} times in {rel}; it must occur exactly once")
        p.write_text(text.replace(old, new), encoding="utf-8")
        return f"updated {rel}"

    def delete(self, rel: str) -> str:
        self._safe(rel).unlink()
        return f"deleted {rel}"

mem = MemoryFS("memory_demo")
print(mem.create("profile.md", "# Dana\n- looks after coastal site CST-7\n"))
print(mem.update("profile.md", "CST-7\n", "CST-7\n- prefers metric units\n"))
print(mem.view("profile.md"))
for bad in ["../outside.md", "/etc/passwd", "notes/../../escape.md"]:
    try:
        mem.view(bad)
    except (ValueError, FileNotFoundError) as e:
        print("blocked:", e)

Expected output:

```text
saved profile.md (40 chars)
updated profile.md
# Dana
- looks after coastal site CST-7
- prefers metric units

blocked: path '../outside.md' is outside the memory folder
blocked: path '/etc/passwd' is outside the memory folder
blocked: path 'notes/../../escape.md' is outside the memory folder
```

Now the tool definition, the loop from lesson "Agents and MCP" with memory as its only tool, and a scripted stand-in for the model so both can be tested offline. The stand-in plays back fixed responses and records every tool result it is shown, which is what the assertions need.

In [ ]:
from types import SimpleNamespace

MEMORY_TOOL = {
    "name": "memory",
    "description": ("Your notes folder, which persists between conversations. 'view' with an empty path lists "
                    "notes; 'view' with a path reads one; 'create' writes a note (overwriting); 'update' "
                    "replaces one exact snippet; 'delete' removes a note. Keep notes short and factual."),
    "input_schema": {"type": "object",
                     "properties": {"command": {"type": "string", "enum": ["view", "create", "update", "delete"]},
                                    "path": {"type": "string", "description": "Relative path such as 'profile.md'"},
                                    "text": {"type": "string"}, "old": {"type": "string"}, "new": {"type": "string"}},
                     "required": ["command", "path"]},
}
MEMORY_SYSTEM = ("You are a personal assistant for a field engineer. At the start of every conversation, view "
                 "your memory folder and read the notes that look relevant. Save durable facts the user tells "
                 "you about themselves, their site or their preferences, and say so when you do. Do not save "
                 "passing details or secrets. Answer briefly.")

def run_memory_tool(mem: MemoryFS, inp: dict) -> str:
    cmd = inp["command"]
    if cmd == "view":   return mem.view(inp.get("path", ""))
    if cmd == "create": return mem.create(inp["path"], inp.get("text", ""))
    if cmd == "update": return mem.update(inp["path"], inp["old"], inp["new"])
    if cmd == "delete": return mem.delete(inp["path"])
    raise ValueError(f"unknown command {cmd!r}")

def run_memory_agent(model, mem: MemoryFS, user_text: str, max_steps: int = 8) -> str:
    """The tool loop from the agents lesson, with the memory tool as its only tool."""
    messages = [{"role": "user", "content": user_text}]
    for _ in range(max_steps):
        resp = model.create(system=MEMORY_SYSTEM, tools=[MEMORY_TOOL], messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for b in resp.content:
            if b.type != "tool_use":
                continue
            try:
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": run_memory_tool(mem, b.input)})
            except Exception as e:
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": f"Error: {e}", "is_error": True})
        messages.append({"role": "user", "content": results})
    return "Stopped: step limit reached."

class ScriptedModel:
    """Plays back a fixed list of responses and records every tool result it is shown."""
    def __init__(self, script: list):
        self.script, self.seen_results = list(script), []
    def create(self, system, tools, messages):
        if isinstance(messages[-1]["content"], list):
            self.seen_results += [r["content"] for r in messages[-1]["content"]]
        return self.script.pop(0)

def tool_call(name, **inp):
    return SimpleNamespace(type="tool_use", id=f"call_{random.randint(0, 999)}", name=name, input=inp)

def turn(*blocks):
    stop = "tool_use" if any(b.type == "tool_use" for b in blocks) else "end_turn"
    return SimpleNamespace(stop_reason=stop, content=list(blocks))

def say(text):
    return SimpleNamespace(type="text", text=text)

In [ ]:
import shutil
shutil.rmtree("memory_demo", ignore_errors=True)
mem = MemoryFS("memory_demo")

session_1 = ScriptedModel([
    turn(tool_call("memory", command="view", path="")),
    turn(tool_call("memory", command="create", path="profile.md",
                   text="# Dana\n- looks after coastal site CST-7\n- prefers metric units\n")),
    turn(say("Nice to meet you, Dana. I have saved that you look after CST-7 and prefer metric units.")),
])
print(run_memory_agent(session_1, mem, "Hi, I'm Dana. I look after coastal site CST-7 and I prefer metric units."))

session_2 = ScriptedModel([                         # a new conversation: only the notes folder carries over
    turn(tool_call("memory", command="view", path="")),
    turn(tool_call("memory", command="view", path="profile.md")),
    turn(say("Metric units: your notes say you prefer them for site CST-7.")),
])
print(run_memory_agent(session_2, mem, "Which units should the report for my site use?"))

assert session_1.seen_results[0] == "(no notes yet)"
assert session_2.seen_results[0] == "profile.md" and "metric" in session_2.seen_results[1]
assert "metric" in mem.view("profile.md")
print("memory survived between sessions")

Expected output:

```text
Nice to meet you, Dana. I have saved that you look after CST-7 and prefer metric units.
Metric units: your notes say you prefer them for site CST-7.
memory survived between sessions
```

With a real model, the script is replaced by the model's own decisions. The first session deliberately includes a password, to see whether the write policy in the system prompt holds.

In [ ]:
# Needs an API key (see the setup cell)
class ClaudeModel:
    def create(self, system, tools, messages):
        return client.messages.create(model=MODEL, max_tokens=1_500, system=system, tools=tools, messages=messages)

shutil.rmtree("memory_live", ignore_errors=True)
live_mem = MemoryFS("memory_live")
print(run_memory_agent(ClaudeModel(), live_mem, "Hi, I'm Dana. I look after coastal site CST-7 and I prefer metric units. "
                                                 "Also, my password is hunter2, don't forget it."))
print("--- notes after session 1:", live_mem.view(""))
print(run_memory_agent(ClaudeModel(), live_mem, "Which units should the report for my site use?"))
# Look for: a view call first, a note written for the durable facts, no password in any note, and a second
# session that reads the note before answering. If the password was saved, tighten MEMORY_SYSTEM and test again.

### Hands-on: provenance, forgetting and an A/B test

1. Subclass `MemoryFS` so every note starts with a header recording its `source` (`user` or `tool`) and the date it was last updated, and so a write whose source is `tool` is refused unless explicitly confirmed.
2. Add a `forget(path)` method for the user's explicit request, and make sure it deletes the file.
3. Write three two-session scenarios (a fact in session one, a question in session two), run them with memory on and off (scripted model offline, real model with a key) and report how many second-session answers are correct.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/16-context-engineering.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.